# 00 · Validate the input and sampling design

**ScGeo 0.2.0 learning track.** Learn what is required before running a sample-aware analysis. This deterministic four-sample fixture includes sparse and condition-only states.

Use the [separate tutorial environment](README.md). Run cells in order in a fresh kernel. Outputs are written to a new ignored results directory; manuscript inputs and outputs are not used.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import scgeo as sg
from IPython.display import display

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents)
             if (p / "scripts/scgeo_v020_examples/SOURCE_PART1.json").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from the scgeo-notebooks checkout.")
examples = root / "scripts/scgeo_v020_examples"
if str(examples) not in sys.path:
    sys.path.insert(0, str(examples))  # only tutorial helpers, not package source
from tutorial_support import setup, save_preview, show_gallery, FORMATS
out = setup(root, "00_input_contract")

## Read-only input preflight

Inspect the declared conditions, state labels, sample IDs, representation and graph provenance. A state missing from a sample does not acquire an imputed cell. The contract checks coverage; it is not a biological validation or a test of significance.

In [ ]:
from deployment_smoke import synthetic_design, analyze, exact_assignments
adata, config = synthetic_design()
display(adata.obs.groupby(["state", "condition", "sample"], observed=True).size().rename("cells").reset_index())
contract = sg.get.input_contract(adata, config=config)
assert contract["ok"] and contract["eligible_states"] == ["shared"]
display(pd.DataFrame(contract["state_coverage"]))
(out / "input_contract.json").write_text(json.dumps(contract, indent=2) + "\n")

## Estimate supported geometry

The helper calls the actual robust-shift, representation-stability and local-geometry APIs. Rotated coordinates are a dependent sensitivity view, not independent biological evidence.

In [ ]:
report = analyze(adata)
display(report)
report.to_csv(out / "state_report.csv", index=False)
np.testing.assert_allclose(adata.uns["scgeo"]["robust_shift"]["by"]["shared"]["delta"], [1., 0.], atol=1e-6)
result = sg.pl.state_evidence_panel(report, show_sample_counts=True, show_inference=True,
                                   return_result=True, show=False)
save_preview(result, out, "input_coverage")

## The four-sample inference limit

Two independent samples per condition have only six possible 2-vs-2 allocations; complementary allocations have identical unsigned displacement. Enumerate rather than permuting cells. This is a pedagogical calculation, not a new package significance test.

In [ ]:
assignments = exact_assignments(adata)
assert assignments["n_assignments"] == 6
display(pd.DataFrame(assignments["assignments"]))
(out / "exact_assignments.json").write_text(json.dumps(assignments, indent=2) + "\n")

**Adaptation:** substitute a documented study design and primary representation in the input contract. For exact installed-source verification rather than a version-pinned tutorial, see the [release deployment preflight](https://github.com/liuifrec/scgeo/blob/v0.2.0/docs/JUPYTERHUB_PREFLIGHT.md).